# Entrenamiento YOLOv8 — Logo Placement


In [1]:
from google.colab import files

uploaded = files.upload()

Saving dataset.zip to dataset.zip


In [ ]:

import zipfile
import os

ZIP_PATH = '/content/dataset.zip'

# Carpeta donde se extraerá
EXTRACT_TO = '/content/proyecto_vision'

os.makedirs(EXTRACT_TO, exist_ok=True)

with zipfile.ZipFile(ZIP_PATH, 'r') as zf:
    zf.extractall(EXTRACT_TO)

print('Dataset descomprimido en', EXTRACT_TO)

Dataset descomprimido en /content/proyecto_vision


In [ ]:
# Instalar dependencias
!pip install ultralytics -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 48.3 MB/s eta 0:00:00


In [ ]:
import torch
print('CUDA disponible:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))

CUDA disponible: True
GPU: Tesla T4


In [ ]:
# Crear data.yaml con rutas correctas para Colab
import yaml

DATASET_PATH = f'{EXTRACT_TO}/dataset'

data_config = {
    'path': DATASET_PATH,
    'train': 'images/train',
    'val': 'images/val',
    'nc': 2,
    'names': ['bottles', 'tshirts']
}

yaml_path = f'{DATASET_PATH}/data_colab.yaml'
with open(yaml_path, 'w') as f:
    yaml.dump(data_config, f)

print('data.yaml creado:', yaml_path)

data.yaml creado: /content/proyecto_vision/dataset/data_colab.yaml


In [ ]:
from ultralytics import YOLO

model = YOLO('yolov8n.pt')

results = model.train(
    data=yaml_path,
    epochs=150,
    imgsz=640,
    batch=16,      # Con GPU T4 se puede usar batch=16 o 32
    device=0,      # GPU
    workers=2,
    patience=30,
    optimizer='AdamW',
    lr0=0.001,
    mosaic=1.0,
    mixup=0.1,
    fliplr=0.5,
    project='/content/runs/train',
    name='logo_placement',
    exist_ok=True,
    plots=True,
)

print('mAP50   :', results.results_dict.get('metrics/mAP50(B)'))
print('mAP50-95:', results.results_dict.get('metrics/mAP50-95(B)'))

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
Ultralytics 8.4.53 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/proyecto_vision/dataset/data_colab.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=150, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0

In [ ]:
import shutil

SAVE_DIR = '/content/drive/MyDrive/logo_placement_model'
shutil.copytree('/content/runs/train/logo_placement', SAVE_DIR, dirs_exist_ok=True)
print('Pesos guardados en Drive:', SAVE_DIR)

Pesos guardados en Drive: /content/drive/MyDrive/logo_placement_model


In [ ]:
best_model = YOLO('/content/runs/train/logo_placement/weights/best.pt')
val_results = best_model.val(data=yaml_path, device=0)

print('Precisión  :', val_results.results_dict.get('metrics/precision(B)'))
print('Recall     :', val_results.results_dict.get('metrics/recall(B)'))
print('mAP50      :', val_results.results_dict.get('metrics/mAP50(B)'))

Ultralytics 8.4.53 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
Model summary (fused): 73 layers, 3,006,038 parameters, 0 gradients, 8.1 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 1916.2±582.1 MB/s, size: 83.5 KB)
val: Scanning /content/proyecto_vision/dataset/labels/val.cache... 49 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 49/49 18.7Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 4/4 3.0it/s 1.3s
                   all         49         49       0.41      0.335       0.36      0.152
               bottles         32         32      0.424      0.375      0.385       0.18
               tshirts         17         17      0.396      0.294      0.334      0.123
Speed: 7.6ms preprocess, 10.0ms inference, 0.0ms loss, 1.2ms postprocess per image
Results saved to /content/runs/detect/val
Precisión  : 0.4102046346770343
Recall     : 0.3345588235294118
mAP50      : 0.35989370766634

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive
